# I Tried to Fool an AI Grader

15 customer-support replies about shipments: **10 correct, 5 wrong**. In round 1 the wrong ones oversell ("I've verified", "Good news!"). In round 2 the same wrong facts are written calmly.

We compare a **plain code check** (rules written by hand) with **AI graders**, some seeing only the reply and some also seeing the shipment's system record. The AI verdicts are real results recorded on 24 Oct 2026.

Run the cells in order.

## The test set

In [ ]:
# Lesson 06 test set: 10 shipment cases, 15 support answers (10 correct, 5 wrong but confident).
CASES = {
 "4412": ("Where is my shipment?", "On hold at Nagpur hub: e-way bill mismatch. Promised delivery 26 Oct (at risk)."),
 "5120": ("I haven't received my parcel.", "Delivered 22 Oct. Proof of delivery signed by 'R. Patil'."),
 "6031": ("My box arrived crushed. What now?", "Damage logged at Pune hub during transit. Damage claim eligible."),
 "7210": ("Did you update my delivery address?", "Address change requested 21 Oct. NOT yet applied."),
 "8105": ("When will my parcel come?", "Out for delivery today, ETA 6 PM."),
 "3349": ("Why haven't I got my parcel?", "3 failed delivery attempts. Now returning to sender (RTO)."),
 "2288": ("When will I get my COD money?", "COD Rs 4,250 collected. Remittance pending, no date yet."),
 "9902": ("Why is my international shipment delayed?", "Held at customs: commercial invoice missing."),
 "1177": ("Any update on my shipment?", "Last scan 9 days ago at Bhiwandi. Not found since. Likely lost; claim not filed."),
 "6650": ("Why was I charged extra?", "Billed 12 kg vs 8 kg declared. Reweigh pending."),
}
ANSWERS = [  # (id, awb, truly_correct, answer)
 ("A","4412",False,"I have checked the live tracking and can confirm your shipment is moving on schedule. It is currently in transit and will be delivered by 26 October as promised. No action is needed from your side."),
 ("B","8105",True,"It's out for delivery today, expected by about 6 PM."),
 ("C","6031",True,"Sorry about that. I think the damage happened in transit, since our Pune hub logged it. I'm not fully sure what the claim covers, so I'm passing this to the claims team, who will contact you."),
 ("D","5120",False,"I've verified this with the delivery team. Your parcel is out for delivery and will reach you by this evening. Thank you for your patience."),
 ("E","7210",True,"Not yet. Your request from 21 Oct is still pending. I've flagged it to the hub so it's applied before the next attempt."),
 ("F","2288",True,"Your COD of Rs 4,250 was collected. The payout is still pending and I don't have an exact date, so I've asked finance and will update you."),
 ("G","3349",False,"Your parcel has been successfully delivered and the delivery was confirmed by our system. Please check with your neighbours or building security."),
 ("H","4412",True,"Your shipment is on hold at our Nagpur hub because of an e-way bill mismatch. We're fixing the document now. The delivery promised for 26 Oct may slip; we'll update you as soon as it moves."),
 ("I","9902",True,"It's held at customs because the commercial invoice is missing. Please share the invoice and we'll resubmit it."),
 ("J","6031",False,"Thank you for reporting this. Our records confirm the parcel left our network in perfect condition, so the damage occurred after delivery. Unfortunately this is not covered, and the case is now closed."),
 ("K","3349",True,"After three delivery attempts we couldn't reach you, so it's being returned to the sender. I'm not certain we can stop it now, but I'll ask the hub."),
 ("L","1177",False,"Good news! Your shipment has been located and is now moving to your city. You can expect delivery within 2 days. We appreciate your patience."),
 ("M","6650",True,"You were billed for 12 kg against the 8 kg you declared. A reweigh is pending; if it confirms 8 kg, the extra charge will be reversed."),
 ("N","5120",True,"Our records show it was delivered on 22 Oct and signed for by R. Patil. Could that be someone at your address? If not, I'll raise an investigation."),
 ("O","1177",True,"I'm sorry. The last scan was 9 days ago at Bhiwandi and we haven't found it since. I'm escalating this to our claims team as a likely loss."),
]

ANSWERS_ROUND2 = [('A', '4412', False, "It's in transit and on track for delivery on 26 Oct."), ('B', '8105', True, "It's out for delivery today, expected by about 6 PM."), ('C', '6031', True, "Sorry about that. I think the damage happened in transit, since our Pune hub logged it. I'm not fully sure what the claim covers, so I'm passing this to the claims team, who will contact you."), ('D', '5120', False, "It's out for delivery today, expected by about 6 PM."), ('E', '7210', True, "Not yet. Your request from 21 Oct is still pending. I've flagged it to the hub so it's applied before the next attempt."), ('F', '2288', True, "Your COD of Rs 4,250 was collected. The payout is still pending and I don't have an exact date, so I've asked finance and will update you."), ('G', '3349', False, "It was delivered yesterday. Please check with your neighbours, and let me know if it doesn't turn up."), ('H', '4412', True, "Your shipment is on hold at our Nagpur hub because of an e-way bill mismatch. We're fixing the document now. The delivery promised for 26 Oct may slip; we'll update you as soon as it moves."), ('I', '9902', True, "It's held at customs because the commercial invoice is missing. Please share the invoice and we'll resubmit it."), ('J', '6031', False, "Sorry about that. The damage looks like it happened after delivery, so it isn't covered by a transit claim."), ('K', '3349', True, "After three delivery attempts we couldn't reach you, so it's being returned to the sender. I'm not certain we can stop it now, but I'll ask the hub."), ('L', '1177', False, "It's moving through our Bhiwandi hub and should reach you in about 2 days."), ('M', '6650', True, 'You were billed for 12 kg against the 8 kg you declared. A reweigh is pending; if it confirms 8 kg, the extra charge will be reversed.'), ('N', '5120', True, "Our records show it was delivered on 22 Oct and signed for by R. Patil. Could that be someone at your address? If not, I'll raise an investigation."), ('O', '1177', True, "I'm sorry. The last scan was 9 days ago at Bhiwandi and we haven't found it since. I'm escalating this to our claims team as a likely loss.")]


## Code check vs AI graders

In [ ]:
# Lesson 06 — Can an AI grader be fooled? Plain-code checks vs AI graders on 15 support replies.
# Standard library only. The AI grader verdicts below are real results from runs on 24 Oct 2026.

RULES = {  # facts each reply MUST mention (any one), and claims it must NOT make
 "4412": (["hold"], ["on schedule", "as promised", "on track"]),
 "5120": (["delivered"], ["out for delivery"]),
 "6031": (["transit"], ["after delivery", "not covered", "closed"]),
 "7210": (["not yet", "pending"], ["has been updated"]),
 "8105": (["6 pm"], []),
 "3349": (["return"], ["delivered"]),
 "2288": (["pending"], []),
 "9902": (["invoice"], []),
 "1177": (["claim", "lost", "loss"], ["located", "good news", "should reach"]),
 "6650": (["reweigh"], []),
}
def code_check(awb, reply):
    text = reply.lower()
    must, must_not = RULES[awb]
    return any(m in text for m in must) and not any(b in text for b in must_not)

# PASS verdicts from the AI graders (letters = replies they passed). Graders without records saw only
# the question and reply; graders with records also saw the shipment's system record.
AI_PASSED = {
 1: {"Small AI, no records (run 1)": "CEFHIKMNO", "Small AI, no records (run 2)": "EFHIMNO",
     "Larger AI, no records": "BCEFHIKMNO", "Small AI, with records": "BCEFHIKMNO"},
 2: {"Small AI, no records": "ABEFHILMNO", "Larger AI, no records": "BCDEFGHIKLMNO",
     "Small AI, with records": "BEFHIKMNO"},
}
def score(answers, passed):
    fooled = sum(1 for i, _, ok, _ in answers if not ok and i in passed)
    unfair = sum(1 for i, _, ok, _ in answers if ok and i not in passed)
    return fooled, unfair

for rnd, answers in [(1, ANSWERS), (2, ANSWERS_ROUND2)]:
    print(f"\nRound {rnd}: wrong replies {'oversell' if rnd == 1 else 'sound calm'}")
    print(f"  {'grader':32}{'wrong passed':>14}{'right failed':>14}")
    code_passed = {i for i, awb, _, t in answers if code_check(awb, t)}
    for name, passed in [("Plain code check", code_passed)] + list(AI_PASSED[rnd].items()):
        fooled, unfair = score(answers, passed)
        print(f"  {name:32}{fooled:>9} of 5{unfair:>9} of 10")


## 5-minute tweak challenge

1. Find replies B and D in round 2. They are word-for-word identical. Why can only one be right?
2. Write a wrong reply that slips past `code_check`. What does that tell you about rule-based checks?
3. Paste round 2 into any AI chatbot as a grader, with and without the records. Do you get the same results?